# **Практика: SHAP for ML models**

Всем привет! Рады видеть вас на второй практической части модуля. Теперь в копилке наших методов еще один — SHAP.

Из урока «Основы семейства SHAP-методов» вы помните, что **SHAP** — метод адаптивный, от чего он является еще и часто используемым. Однако, у значений Шепли есть еще одна причина, по которой их стоит любить — безумно красивый процесс интерпретации.


Работать будем с набором данных California Housing — калифорнийские дома. Напомним, что каждая строка в наборе данных отражает отдельный жилой район и содержит информацию о средних характеристиках домов в этом районе.

**Основные характеристики:**

**Целевая переменная (target):**

- `MedHouseVal` — медианная стоимость домов в данном районе, выраженная в сотнях тысяч долларов.

**Признаки (features):**

- `MedInc` (Median Income): медианный доход в районе (в десятках тысяч долларов)

- `HouseAge` (Housing Median Age): медианный возраст домов в районе (в годах)
- `AveRooms` (Average Rooms per Dwelling): среднее количество комнат на одно жилье.
- `AveBedrms` (Average Bedrooms per Dwelling): среднее количество спален на одно жилье.
- `Population (Population):` численность населения в районе.
-  `AveOccup (Average Occupancy):` среднее количество жильцов на одно жилище.
-  `Latitude (Latitude):` географическая широта района.
- `Longitude (Longitude):` географическая долгота района.

В отличие от прошлой практической части, будем использовать модель CatBoost. Надеемся, эта практика будет для вас интересной и визуально приятной!

Приступим! Приятного кодинга!


In [ ]:
#установим необходимые библиотеки
!pip install shap catboost -q


In [ ]:
import catboost
import pandas as pd
import numpy as np
import shap
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score


data = pd.read_csv('https://github.com/SadSabrina/open-xai-materials/raw/main/data/fetch_california_housing.csv',
                   index_col=0)

X, y = data.drop('target', axis=1), data['target']

rng = np.random.RandomState(0)
bin_var = pd.Series(rng.randint(0, 1, X.shape[0]), name="rnd_bin")
# два случайных признака: rnd_bin — константа 0, rnd_num — случайная перестановка номеров строк.
# Просто номер строки (np.arange) случайным не был бы: строки набора упорядочены по географии,
# и номер строки стал бы для модели заменой координат.
num_var = pd.Series(rng.permutation(X.shape[0]), name="rnd_num")

X_with_rnd_feat = pd.concat((X, bin_var, num_var), axis=1)


## **Значения Шепли вручную**

Прежде чем строить графики, посчитаем значения Шепли сами, по формуле из теории:

$$Sh(v)_i = \sum_{S \subseteq N\setminus i}\frac{|S|!\,(n-|S|-1)!}{n!}\,\big(v(S\cup i) - v(S)\big).$$

В ячейке ниже — прогнозы модели на всех 8 коалициях из примера урока теории «любит ли девушка цветы»: $v(\{\}) = 0.548$ — пустая коалиция (среднее по листьям), $v(\{accounts\_count, cats, photos\}) = 0.5$ — гранд-коалиция.


**Q1. Игрушечный пример («любит ли девушка цветы», 3 признака: accounts_count, cats, photos) с заданными прогнозами модели на всех 8 коалициях. Посчитайте значение Шепли для `cats` вручную по формуле. Чему оно равно (округлите до десятитысячных)?**

Введите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/5/1) · [шаг на Степике](https://stepik.org/lesson/1330425/step/2)

In [ ]:
from itertools import combinations
from math import factorial

players_flowers = ['accounts_count', 'cats', 'photos']

# прогнозы модели на всех 8 коалициях (ключ — множество признаков в коалиции)
v_flowers = {
    frozenset(): 0.548,
    frozenset({'accounts_count'}): 0.6,
    frozenset({'photos'}): 0.3,
    frozenset({'cats'}): 0.6,
    frozenset({'photos', 'accounts_count'}): 0.4,
    frozenset({'cats', 'accounts_count'}): 0.6,
    frozenset({'photos', 'cats'}): 0.45,
    frozenset({'accounts_count', 'cats', 'photos'}): 0.5,
}

# Ваш код здесь: напишите функцию shapley_value(v, players, i), которая перебирает
# все коалиции S без признака i (combinations) и суммирует вес |S|!(n-|S|-1)!/n!,
# умноженный на v(S ∪ i) - v(S). Затем посчитайте shapley_value(v_flowers, players_flowers, 'cats')


На практике прогнозов на коалициях никто не дает: модель ждет полный вектор признаков. Поэтому $v(S)$ считают через референсную выборку $R$: признаки из $S$ берут у объясняемого объекта $x$, остальные — у каждого объекта $r \in R$, и прогнозы усредняют:

$$v(S) = \frac{1}{|R|}\sum_{r \in R} f\big(x_S,\ r_{\bar S}\big).$$

В ячейке ниже обучена CatBoost-модель на трех признаках, выбраны референсная выборка `R` (50 объектов из обучающей части) и объясняемый объект `x` (первый объект тестовой части).


**Q2. Постройте v(S) через референсную выборку на настоящей CatBoost-модели (3 признака) и сравните ручной расчет значений Шепли с `shap.explainers.Exact` на том же R и x. Какой порядок расхождения?**

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/5/2) · [шаг на Степике](https://stepik.org/lesson/1330425/step/3)

In [ ]:
features_3 = ['MedInc', 'HouseAge', 'AveRooms']

X3_train, X3_test, y3_train, y3_test = train_test_split(X[features_3], y, random_state=42)

model_3 = CatBoostRegressor(iterations=300, learning_rate=0.1, random_seed=123)
model_3.fit(X3_train, y3_train, verbose=False, plot=False)

R = X3_train.sample(50, random_state=0)  # референсная выборка
x = X3_test.iloc[[0]]                    # объясняемый объект


In [ ]:
# Ваш код здесь:
# 1) функция v(S): копия R, в которой столбцы из S заменены значениями x, и средний прогноз model_3 по этой копии;
# 2) значения Шепли для трех признаков по формуле (функция из Q1, только v теперь считается через R);
# 3) explainer_exact = shap.explainers.Exact(model_3.predict, shap.maskers.Independent(R, max_samples=len(R)))
#    и explainer_exact(x).values[0];
# 4) максимум модуля разности двух ответов.


Как прежде — сразу обучим модель на имеющихся данных. Однако, в этот раз мы пропускаем процесс нормализации признаков.


**Q3. Почему при обучении CatBoost можно пропустить нормализацию признаков?**

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/5/3) · [шаг на Степике](https://stepik.org/lesson/1330425/step/4)

In [ ]:
#разобьем данные на train и test

X_train, X_test, y_train, y_test = train_test_split(
    X_with_rnd_feat, y, random_state=42
)

model = CatBoostRegressor(iterations=300, learning_rate=0.1, random_seed=123)
model.fit(X_train, y_train, verbose=False, plot=False)

model.best_score_


## **Анализ индивидуальных прогнозов при помощи SHAP**


1. **Force plot.**

Дословно данный тип графика отражает *силу* влияния конкретного значения признака на прогноз модели. Для большей детализации график окрашен голубым (понижение прогноза модели) и красным (увеличение прогноза модели) цветами.


**Для любого одного наблюдения force plot позволяет увидеть:**
- среднее значение целевого признака (вернее, матожидание) *по изначальным входным данным*
- значения конкретных переменных, участвовавших в прогнозировании целевого значения для признака
- силу и направление вклада конкретных значений признаков на изменение прогноза от среднего


In [ ]:
shap.initjs()
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_train)

# Визуализируем прогнозы для 7-го объекта в тренировочных данных
shap.plots.force(shap_values[7, :])


Детализировать force plot до конкретных значений помогает waterfall plot.


Единственный минус `force_plot` — невозможность оценить конкретные значения Шепли. Однако, чтобы устранить это, мы можем детализировать `force_plot` при помощи `waterfall_plot`


In [ ]:
# Визуализируем прогнозы для 7-го объекта в тренировочных данных
shap.plots.waterfall(shap_values[7, :], max_display=5)


Или `bar` plot.


In [ ]:
shap.plots.bar(shap_values[7], max_display=5)


In [ ]:
# SHAP-значения для тестового набора: этот объяснитель и эти значения понадобятся во всех заданиях ниже
test_explainer = shap.TreeExplainer(model)
test_shap_values = test_explainer(X_test)


**Q4. Постройте `force`-график для объекта с индексом 7 из тестового набора. Чему равен прогноз модели (округлите до сотых)?**

Введите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/5/4) · [шаг на Степике](https://stepik.org/lesson/1330425/step/5)

In [ ]:
shap.initjs()

# Ваш код здесь


Если график выше повернуть на 90 градусов и сложить, то получится следующая картинка


In [ ]:
shap.initjs()
shap.plots.force(shap_values[7:9,]) #сложенный график для двух наблюдений


Для больших бессвязных сегментов в наборе данных график не очень информативен (ради интереса, попробуйте получить график для большого количества значений или обратитесь к примеру ниже). С точки зрения практики здесь для нас оказывается полезным то, что этот график может оказаться полезным при рассмотрении отдельных признаков.


## **Анализ глобального поведения модели при помощи SHAP**


In [ ]:
shap.initjs()

#Посмотрим на график для первых 100 объектов тренировочного набора данных
shap.plots.force(shap_values[:100])


Выбирая отдельные шкалы на осях x и y, мы можем увидеть основную тенденцию влияния признака для всех наборов данных. Например, выбрав слева и сверху MedInc effects и MedInc соответственно, можно увидеть, что данный признак способен как увеличивать, так и уменьшать прогноз от его матожидания.


**Q5a. Постройте «стек» force-plot для первых 100 объектов теста. Какой признак чаще всего *увеличивает* прогноз от базового?**

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/5/5) · [шаг на Степике](https://stepik.org/lesson/1330425/step/6)

In [ ]:
shap.initjs()

#Ваш код здесь


**Q5b. Тот же «стек». Какой признак у наибольшего числа объектов *уменьшает* прогноз от базового?**

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/5/6) · [шаг на Степике](https://stepik.org/lesson/1330425/step/7)

In [ ]:
# Ваш код здесь: по тому же стеку (или по test_shap_values.values[:100]) посчитайте,
# у какого признака вклад чаще всего отрицательный


**Q6. Оказывает ли на модель влияние хоть один из случайных признаков (rnd_bin, rnd_num)? Сравните mean|SHAP| обоих с самым слабым «настоящим» признаком (Population).**

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/5/7) · [шаг на Степике](https://stepik.org/lesson/1330425/step/8)

In [ ]:
# Ваш код здесь: mean|SHAP| каждого признака по test_shap_values


Дальше, мы можем попробовать посмотреть, существует ли явное парное взаимодействие признаков. Оно называется *feature interaction*.

Взаимодействием признаков называют комбинированный эффект двух или более признаков на целевую переменную. Другими словами, взаимодействие происходит, когда эффект одного признака на целевую переменную зависит от значения другого признака.

Пример парного взаимодействия — зависимость цены дома от площади. Например, большая площадь может вести к бОльшей цене, но явный эффект такого поведения наблюдается только для молодых домов.  


In [ ]:
# Создадим dependence plot для рассмотрения влияния пары признаков на выход модели
shap.dependence_plot("AveRooms", shap_values.values[:100], X_train[:100], interaction_index="HouseAge")


Разберемся с осями графика.
- Левая шкала отражает значение Шепли. Внимательно смотрите на диапазоны изменения значений, чтобы не извлечь ошибочный вывод. Например, может показаться, что для старших домов маленькое среднее количество комнат снижает цену, но это не так. На самом деле, чем старше дом, тем меньше эффекта от среднего количества комнат на дом.
- Правая и нижняя шкалы отражают признаки, взаимодействие которых мы рассматриваем.



**Сопоставьте величину с тем, где она показана на `dependence_plot("AveRooms", …, interaction_index="HouseAge")`**

Выполните сопоставление в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/5/8) · [шаг на Степике](https://stepik.org/lesson/1330425/step/9)

**Постройте `dependence_plot` для тех же признаков, для первых 100 объектов, но на основе test_explainer и test_shap_values. Выберите верные утверждения:**

Выберите все верные утверждения в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/5/9) · [шаг на Степике](https://stepik.org/lesson/1330425/step/10)

In [ ]:
# Ваш код здесь


Итак, мы с вами рассмотрели влияния одного и комбинации из двух признаков на поведение модели. Теперь рассмотрим логическое завершение — средний вклад каждого в прогноз.  


In [ ]:
# summarize the effects of all the features
shap.plots.beeswarm(shap_values)


Данный график позволяет получить выводы об обобщенном влиянии каждого признака. Так, например мы можем увидеть, что:
- Широта и долгота (расположение) увеличиваясь ведут к уменьшению прогноза;
- Чем больше медианный доход, тем сильнее повышается вклад в медианную стоимость домов.
- Большие значения Population практически не оказывают влияния на модель
- Большое среднее количество спален способно немного влиять на модель


**Постройте `summary plot` (`shap.plots.beeswarm`) для тестовых данных, как в тетради выше для обучающих. Остаются ли верными для тестового набора выводы, сделанные по обучающим данным?**

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/5/10) · [шаг на Степике](https://stepik.org/lesson/1330425/step/11)

In [ ]:
#Ваш код здесь


Иной способ представления суммарной важности, ее представление в виде столбчатой диаграммы. Обратите внимание, что на ней берется среднее значение по **модулю коэффициентов** Шепли.


In [ ]:
# summarize the effects of all the features
shap.plots.bar(shap_values)


**Какая информация теряется при анализе столбчатого графика?**

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/5/11) · [шаг на Степике](https://stepik.org/lesson/1330425/step/12)

И последний тип визуализации на сегодня — тепловая карта. Ее мы приводим просто для ознакомления, так как не зная предыдущей логики графиков, читать ее — сложно. Однако, на основе изученного можно видеть, что такая карта — хорошее обобщение для почти всех возможных выводов из коэффициентов Шепли.


In [ ]:
# визуализация в виде тепловой карты
shap.plots.heatmap(shap_values[:100])


## **Какой объяснитель выбирает библиотека**

`shap.Explainer` сам выбирает алгоритм по тому, что ему передали. Если передать функцию прогноза (например, `model.predict`), библиотека ничего не знает об устройстве модели: при малом числе признаков (примерно до 10) она выбирает `ExactExplainer` — полный перебор всех $2^n$ коалиций, при большом — `PermutationExplainer`, приближение через случайные перестановки. Если же передать саму модель, библиотека может узнать ее тип.


**Q7. Передайте в `shap.Explainer` саму модель CatBoost (не `.predict`) при 10 признаках. Какой класс Explainer будет выбран?**

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/5/12) · [шаг на Степике](https://stepik.org/lesson/1330425/step/13)

In [ ]:
print('n_features =', X_train.shape[1])

# Ваш код здесь: передайте model в shap.Explainer и выведите type() получившегося объекта


Значение Шепли можно записать и через порядки прихода признаков: признаки по одному добавляют в «команду» в каком-то порядке, записывают прирост каждого в момент его прихода и усредняют по всем $n!$ порядкам.

Пусть модель оценивает квартиру в тысячах у.е., признаков три: **A** — площадь, **D** — район, **F** — этаж. Каждый прогноз строим так: известные признаки фиксируем на значениях нашей квартиры, а неизвестные подставляем от случайных чужих квартир из данных и усредняем. Пустое множество — просто средний прогноз по рынку. Прогнозы на всех 8 коалициях — в ячейке ниже (в коде A = `area`, D = `district`, F = `floor`).


**Q8. Пример «квартира» (площадь A / район D / этаж F, 3 признака, известны прогнозы на всех 8 коалициях). Прогнозы модели на коалициях: $v(\varnothing)=100$, $v(A)=130$, $v(D)=120$, $v(F)=105$, $v(AD)=170$, $v(AF)=135$, $v(DF)=125$, $v(ADF)=180$. Чему равно точное значение Шепли для площади, $\phi_A$? Ответ округлите до десятых.**

Введите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/5/13) · [шаг на Степике](https://stepik.org/lesson/1330425/step/14)

In [ ]:
players_flat = ['area', 'district', 'floor']

# прогнозы модели на всех 8 коалициях, тысячи у.е.
v_flat = {
    frozenset(): 100,
    frozenset({'area'}): 130,
    frozenset({'district'}): 120,
    frozenset({'floor'}): 105,
    frozenset({'area', 'district'}): 170,
    frozenset({'area', 'floor'}): 135,
    frozenset({'district', 'floor'}): 125,
    frozenset({'area', 'district', 'floor'}): 180,
}

# Ваш код здесь: значение Шепли для 'area' (подойдет функция из Q1 или перебор всех 3! порядков)


В библиотеке есть несколько способов посчитать одни и те же значения Шепли: `LinearExplainer` (точная формула для линейной модели), `ExactExplainer` (полный перебор коалиций), `KernelExplainer` (взвешенная регрессия на коалициях) и `SamplingExplainer` (метод IME Štrumbelj и Kononenko: оценка Монте-Карло по случайным перестановкам). В ячейке ниже обучена линейная регрессия на восьми настоящих признаках, выбраны референсная выборка `R_lin` (100 объектов из обучающей части) и объясняемый объект `x_lin`.


**Q9. Сравните `LinearExplainer`, `ExactExplainer`, `KernelExplainer`, `SamplingExplainer` на одной линейной модели и одном объекте. Какой из них дает заметно отличающийся результат — и почему?**

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/5/14) · [шаг на Степике](https://stepik.org/lesson/1330425/step/15)

In [ ]:
from sklearn.linear_model import LinearRegression

real_features = list(X.columns)  # восемь настоящих признаков, без случайных

lin_model = LinearRegression().fit(X_train[real_features], y_train)

R_lin = X_train[real_features].sample(100, random_state=0)  # референсная выборка
x_lin = X_test[real_features].iloc[[0]]                      # объясняемый объект


In [ ]:
# Ваш код здесь: значения Шепли для x_lin четырьмя способами на одной и той же R_lin
# shap.LinearExplainer(lin_model, shap.maskers.Independent(R_lin, max_samples=100))(x_lin).values[0]
# shap.explainers.Exact(lin_model.predict, shap.maskers.Independent(R_lin, max_samples=100))(x_lin).values[0]
# shap.KernelExplainer(lin_model.predict, R_lin).shap_values(x_lin)[0]
# shap.SamplingExplainer(lin_model.predict, R_lin).shap_values(x_lin, nsamples=...)[0]  — попробуйте разные nsamples
# Сравните четыре вектора между собой.
